# Spam Text Classification

**Student Name:**  
**Student ID:**  
**AI Usage Declaration:**  

Minimal setup and reusable helpers for the project.

In [25]:
import re
from collections import Counter
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

Manual metrics used for cross-validation and final evaluation.

In [26]:
def accuracy_score_manual(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(y_true == y_pred))


def precision_score_manual(y_true, y_pred, positive_label="spam"):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = np.sum((y_true == positive_label) & (y_pred == positive_label))
    fp = np.sum((y_true != positive_label) & (y_pred == positive_label))
    denominator = tp + fp
    return float(tp / denominator) if denominator else 0.0


def recall_score_manual(y_true, y_pred, positive_label="spam"):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = np.sum((y_true == positive_label) & (y_pred == positive_label))
    fn = np.sum((y_true == positive_label) & (y_pred != positive_label))
    denominator = tp + fn
    return float(tp / denominator) if denominator else 0.0


def f1_score_manual(y_true, y_pred, positive_label="spam"):
    precision = precision_score_manual(y_true, y_pred, positive_label=positive_label)
    recall = recall_score_manual(y_true, y_pred, positive_label=positive_label)
    denominator = precision + recall
    return float((2 * precision * recall) / denominator) if denominator else 0.0


def macro_f1_score(y_true, y_pred, labels=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if labels is None:
        labels = sorted(np.unique(np.concatenate([y_true, y_pred])))
    scores = [f1_score_manual(y_true, y_pred, positive_label=label) for label in labels]
    return float(np.mean(scores)) if scores else 0.0


def false_positive_rate(y_true, y_pred, positive_label="spam"):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fp = np.sum((y_true != positive_label) & (y_pred == positive_label))
    tn = np.sum((y_true != positive_label) & (y_pred != positive_label))
    denominator = fp + tn
    return float(fp / denominator) if denominator else 0.0


def metric_summary(y_true, y_pred, positive_label="spam"):
    labels = sorted(np.unique(np.concatenate([np.asarray(y_true), np.asarray(y_pred)])))
    return {
        "accuracy": accuracy_score_manual(y_true, y_pred),
        "precision": precision_score_manual(y_true, y_pred, positive_label=positive_label),
        "recall": recall_score_manual(y_true, y_pred, positive_label=positive_label),
        "f1_score": f1_score_manual(y_true, y_pred, positive_label=positive_label),
        "macro_f1_score": macro_f1_score(y_true, y_pred, labels=labels),
        "false_positive_rate": false_positive_rate(y_true, y_pred, positive_label=positive_label),
    }

Balance the full dataset first, then create the train and test split.

In [27]:


def random_undersample(df, label_col="Category", random_state=42):
    rng = np.random.default_rng(random_state)
    class_counts = df[label_col].value_counts()
    target_count = class_counts.min()

    balanced_parts = []
    for label in sorted(class_counts.index):
        class_indices = df.index[df[label_col] == label].to_numpy().copy()
        chosen_indices = rng.choice(class_indices, size=target_count, replace=False)
        balanced_parts.append(df.loc[chosen_indices])

    balanced_df = pd.concat(balanced_parts, axis=0)
    balanced_df = balanced_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return balanced_df


def stratified_train_test_split(df, label_col="Category", test_size=0.2, random_state=42):
    rng = np.random.default_rng(random_state)
    train_parts = []
    test_parts = []

    for label in sorted(df[label_col].unique()):
        class_df = df[df[label_col] == label].sample(frac=1, random_state=random_state)
        
        class_indices = class_df.index.to_numpy().copy()
        rng.shuffle(class_indices)

        n_test = max(1, int(np.floor(len(class_indices) * test_size)))
        test_idx = class_indices[:n_test]
        train_idx = class_indices[n_test:]

        test_parts.append(df.loc[test_idx])
        train_parts.append(df.loc[train_idx])

    train_df = pd.concat(train_parts, axis=0).sample(frac=1, random_state=random_state).reset_index(drop=True)
    test_df = pd.concat(test_parts, axis=0).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return train_df, test_df


data_path = Path("SPAM text message 20170820 - Data.csv")
if not data_path.exists():
    data_path = Path("spam_mails") / "SPAM text message 20170820 - Data.csv"

df = pd.read_csv(data_path)
balanced_df = random_undersample(df, label_col="Category", random_state=SEED)
train_df, test_df = stratified_train_test_split(
    balanced_df,
    label_col="Category",
    test_size=0.2,
    random_state=SEED,
)

print("Original shape:", df.shape)
print("Balanced shape:", balanced_df.shape)
print("Balanced class counts:")
print(balanced_df["Category"].value_counts())
print()
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train class counts:")
print(train_df["Category"].value_counts())

Original shape: (5572, 2)
Balanced shape: (1494, 2)
Balanced class counts:
Category
spam    747
ham     747
Name: count, dtype: int64

Train shape: (1196, 2)
Test shape: (298, 2)
Train class counts:
Category
spam    598
ham     598
Name: count, dtype: int64


Custom vectorizer and Multinomial Naive Bayes implementation with no sklearn feature tools.

In [28]:
class ScratchVectorizer:
    def __init__(self, representation="count", min_df=1, max_features=None):
        if representation not in {"count", "tfidf"}:
            raise ValueError("representation must be 'count' or 'tfidf'")
        self.representation = representation
        self.min_df = min_df
        self.max_features = max_features
        self.vocabulary_ = None
        self.feature_names_ = None
        self.idf_ = None
        self.n_docs_ = 0

    @staticmethod
    def tokenize(text):
        text = str(text).lower()
        return re.findall(r"[a-z0-9]+(?:'[a-z0-9]+)?", text)

    def fit(self, texts):
        texts = pd.Series(texts).fillna("")
        self.n_docs_ = len(texts)

        document_frequency = Counter()
        total_frequency = Counter()

        for text in texts:
            tokens = self.tokenize(text)
            total_frequency.update(tokens)
            document_frequency.update(set(tokens))

        eligible_terms = [
            term for term, df_value in document_frequency.items()
            if df_value >= self.min_df
        ]

        ranked_terms = sorted(
            eligible_terms,
            key=lambda term: (-total_frequency[term], term),
        )

        if self.max_features is not None:
            ranked_terms = ranked_terms[: self.max_features]

        self.feature_names_ = ranked_terms
        self.vocabulary_ = {term: idx for idx, term in enumerate(self.feature_names_)}

        if self.representation == "tfidf":
            self.idf_ = np.array([
                np.log((1 + self.n_docs_) / (1 + document_frequency[term])) + 1.0
                for term in self.feature_names_
            ])
        else:
            self.idf_ = None

        return self

    def transform(self, texts):
        if self.vocabulary_ is None:
            raise ValueError("Vectorizer must be fitted before calling transform.")

        texts = pd.Series(texts).fillna("")
        n_docs = len(texts)
        n_features = len(self.feature_names_)
        X = np.zeros((n_docs, n_features), dtype=float)

        for row_idx, text in enumerate(texts):
            tokens = self.tokenize(text)
            if not tokens:
                continue

            counts = Counter(tokens)
            if self.representation == "count":
                for term, count in counts.items():
                    feature_idx = self.vocabulary_.get(term)
                    if feature_idx is not None:
                        X[row_idx, feature_idx] = count
            else:
                total_terms = sum(counts.values())
                for term, count in counts.items():
                    feature_idx = self.vocabulary_.get(term)
                    if feature_idx is not None:
                        tf_value = count / total_terms
                        X[row_idx, feature_idx] = tf_value * self.idf_[feature_idx]

        return X

    def fit_transform(self, texts):
        self.fit(texts)
        return self.transform(texts)

    def get_feature_names_out(self):
        if self.feature_names_ is None:
            raise ValueError("Vectorizer has not been fitted yet.")
        return np.array(self.feature_names_)


class ScratchMultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes_ = None
        self.class_log_prior_ = None
        self.feature_log_prob_ = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        self.classes_, class_counts = np.unique(y, return_counts=True)
        n_features = X.shape[1]
        self.class_log_prior_ = np.log(class_counts / class_counts.sum())
        self.feature_log_prob_ = np.zeros((len(self.classes_), n_features), dtype=float)

        for class_idx, class_label in enumerate(self.classes_):
            X_class = X[y == class_label]
            smoothed_feature_counts = X_class.sum(axis=0) + self.alpha
            smoothed_total = smoothed_feature_counts.sum()
            self.feature_log_prob_[class_idx] = np.log(smoothed_feature_counts / smoothed_total)

        return self

    def predict_log_proba(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.feature_log_prob_.T + self.class_log_prior_

    def predict(self, X):
        log_scores = self.predict_log_proba(X)
        return self.classes_[np.argmax(log_scores, axis=1)]

Use 5-fold cross-validation on the training split and rank hyperparameters by average macro F1.

In [29]:
def make_stratified_folds(y, n_splits=5, random_state=42):
    y = np.asarray(y)
    rng = np.random.default_rng(random_state)
    unique_labels = np.unique(y)
    per_label_folds = {label: [] for label in unique_labels}

    for label in unique_labels:
        label_indices = np.where(y == label)[0]
        rng.shuffle(label_indices)
        per_label_folds[label] = np.array_split(label_indices, n_splits)

    folds = []
    all_indices = np.arange(len(y))

    for fold_idx in range(n_splits):
        val_indices = np.concatenate([
            per_label_folds[label][fold_idx] for label in unique_labels
        ])
        train_mask = np.ones(len(y), dtype=bool)
        train_mask[val_indices] = False
        train_indices = all_indices[train_mask]
        folds.append((train_indices, val_indices))

    return folds


param_grid = {
    "representation": ["count", "tfidf"],
    "min_df": [2, 5],
    "max_features": [1000, 2000],
    "alpha": [0.5, 1.0, 2.0],
}

param_combinations = [
    dict(zip(param_grid.keys(), values))
    for values in product(*param_grid.values())
]

folds = make_stratified_folds(train_df["Category"].to_numpy(), n_splits=5, random_state=SEED)
cv_results = []

for params in param_combinations:
    fold_scores = []

    for train_idx, val_idx in folds:
        fold_train = train_df.iloc[train_idx].reset_index(drop=True)
        fold_val = train_df.iloc[val_idx].reset_index(drop=True)

        vectorizer = ScratchVectorizer(
            representation=params["representation"],
            min_df=params["min_df"],
            max_features=params["max_features"],
        )

        X_fold_train = vectorizer.fit_transform(fold_train["Message"])
        X_fold_val = vectorizer.transform(fold_val["Message"])

        model = ScratchMultinomialNB(alpha=params["alpha"])
        model.fit(X_fold_train, fold_train["Category"])
        val_pred = model.predict(X_fold_val)

        fold_macro_f1 = macro_f1_score(
            fold_val["Category"].to_numpy(),
            val_pred,
            labels=["ham", "spam"],
        )
        fold_scores.append(fold_macro_f1)

    result_row = {
        **params,
        "avg_macro_f1": float(np.mean(fold_scores)),
        "std_macro_f1": float(np.std(fold_scores)),
    }
    cv_results.append(result_row)

cv_results_df = pd.DataFrame(cv_results).sort_values(
    by=["avg_macro_f1", "std_macro_f1"],
    ascending=[False, True],
).reset_index(drop=True)

best_params = cv_results_df.iloc[0][["representation", "min_df", "max_features", "alpha"]].to_dict()

print("Top CV Results:")
display(cv_results_df.head(10))
print("Best hyperparameters:", best_params)

Top CV Results:


,representation,min_df,max_features,alpha,avg_macro_f1,std_macro_f1
0,count,2,2000,1.0,0.956486,0.010525
1,count,2,2000,2.0,0.956481,0.012117
2,count,2,2000,0.5,0.956480,0.012661
3,tfidf,2,2000,2.0,0.954790,0.011769
4,count,2,1000,0.5,0.953958,0.014873
5,tfidf,2,1000,2.0,0.953954,0.012810
6,tfidf,2,2000,1.0,0.953120,0.012928
7,count,2,1000,2.0,0.953112,0.013984
8,tfidf,2,2000,0.5,0.953110,0.014013
9,count,2,1000,1.0,0.953106,0.014014


Best hyperparameters: {'representation': 'count', 'min_df': 2, 'max_features': 2000, 'alpha': 1.0}


Retrain the best configuration on the full training set, then evaluate once on the untouched test set.

In [30]:
final_vectorizer = ScratchVectorizer(
    representation=best_params["representation"],
    min_df=int(best_params["min_df"]),
    max_features=int(best_params["max_features"]),
)

X_train_final = final_vectorizer.fit_transform(train_df["Message"])
X_test_final = final_vectorizer.transform(test_df["Message"])

final_model = ScratchMultinomialNB(alpha=float(best_params["alpha"]))
final_model.fit(X_train_final, train_df["Category"])

test_predictions = final_model.predict(X_test_final)
final_metric_summary = metric_summary(
    test_df["Category"].to_numpy(),
    test_predictions,
    positive_label="spam",
)

print("Final metric_summary:")
print(final_metric_summary)

prediction_preview = test_df[["Message", "Category"]].copy()
prediction_preview["Prediction"] = test_predictions
prediction_preview.head(5)

Final metric_summary:
{'accuracy': 0.9664429530201343, 'precision': 0.9793103448275862, 'recall': 0.9530201342281879, 'f1_score': 0.9659863945578231, 'macro_f1_score': 0.9664369058881831, 'false_positive_rate': 0.020134228187919462}


,Message,Category,Prediction
0,Want 2 get laid tonight? Want real Dogging locations sent direct 2 ur mob? Join the UK's largest Dogging Network bt ...,spam,spam
1,Please CALL 08712402779 immediately as there is an urgent message waiting for you,spam,spam
2,"Final Chance! Claim ur £150 worth of discount vouchers today! Text YES to 85023 now! SavaMob, member offers mobile! ...",spam,spam
3,"Lemme know when I can swing by and pick up, I'm free basically any time after 1 all this semester",ham,ham
4,U still going to the mall?,ham,ham


Inspect the final model with log-odds ratios to see which words lean most toward spam or ham.

In [31]:
feature_names = final_vectorizer.get_feature_names_out()
class_to_index = {label: idx for idx, label in enumerate(final_model.classes_)}

spam_idx = class_to_index["spam"]
ham_idx = class_to_index["ham"]

log_odds = final_model.feature_log_prob_[spam_idx] - final_model.feature_log_prob_[ham_idx]

spam_words = pd.DataFrame({
    "word": feature_names,
    "log_odds_spam_vs_ham": log_odds,
}).sort_values("log_odds_spam_vs_ham", ascending=False).head(10).reset_index(drop=True)

ham_words = pd.DataFrame({
    "word": feature_names,
    "log_odds_spam_vs_ham": log_odds,
}).sort_values("log_odds_spam_vs_ham", ascending=True).head(10).reset_index(drop=True)

print("Top 10 spam-indicative words:")
display(spam_words)
print("Top 10 ham-indicative words:")
display(ham_words)

Top 10 spam-indicative words:


,word,log_odds_spam_vs_ham
0,claim,3.963503
1,www,3.842875
2,prize,3.720273
3,150p,3.629301
4,cash,3.613301
5,won,3.580511
6,tone,3.475150
7,nokia,3.456458
8,text,3.388139
9,co,3.336314


Top 10 ham-indicative words:


,word,log_odds_spam_vs_ham
0,gt,-4.069182
1,lt,-4.040194
2,he,-3.691888
3,lor,-3.649328
4,she,-3.404205
5,amp,-3.347047
6,anything,-3.221884
7,ask,-3.152891
8,haha,-3.152891
9,went,-3.152891
